# 🌌 Vega-1B: Llama-3.2-1B'yi Vega verisiyle ince ayarla (LoRA)

Bu defter, **gerçekten 1.24 milyar parametreli** ve **~9 trilyon token** görmüş Llama-3.2-1B'yi,
Vega'nın Türkçe bilgi tabanı (351 soru-cevap) + 14 MB çok dilli kod derlemiyle **LoRA** yöntemiyle uzmanlaştırır.

**Dürüst çerçeve:** 1B'yi sıfırdan eğitmek Colab'da mümkün değildir (20B token ≈ TPU v5e-1 ile kesintisiz ~3-4 hafta; oturumlar 12-24 saatte kopar). LoRA ince ayar ise ~30-60 dakikadır ve gerçek eğitimdir: ~11 milyon LoRA parametresi gradyanla güncellenir, 1B'lik gövde bilgisini korur.

**Çalışma ortamı:** Runtime → Change runtime type → **GPU (T4 yeter, A100 idealdir)**.
TPU v5e-1 notu: PyTorch+PEFT ekosistemi TPU'da (torch_xla) kırılgandır; TPU'yu JAX/MaxText ile *sıfırdan küçük model* eğitmek için kullanmak mantıklı, ince ayar için GPU çalışma zamanı pragmatik seçimdir. Bu defter GPU'da test edilecek şekilde yazıldı.

Llama-3.2 erişimi için HF hesabınla model sayfasında lisansı onayla ve bir `HF_TOKEN` gir (Colab: sol panel → 🔑 Secrets).

In [ ]:
!pip -q install -U transformers peft datasets accelerate bitsandbytes
import torch
print('cihaz:', 'GPU: ' + torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'GPU YOK — Runtime tipini değiştir')

In [ ]:
# Vega verisini depodan indir
BRANCH = 'claude/vega-ai-platform-g4fadn'
BASE = f'https://raw.githubusercontent.com/sigmagmchess/ai/{BRANCH}/data/'
!wget -q {BASE}vega-instruct.jsonl {BASE}vega-corpus.txt.gz
!gunzip -kf vega-corpus.txt.gz
import json
pairs = [json.loads(l) for l in open('vega-instruct.jsonl')]
corpus = open('vega-corpus.txt').read()
print(len(pairs), 'soru-cevap |', f'{len(corpus)/1e6:.1f}M karakter derlem')

In [ ]:
from google.colab import userdata
import os
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
MODEL = 'meta-llama/Llama-3.2-1B-Instruct'   # alternatif (lisanssız): 'Qwen/Qwen2.5-0.5B-Instruct'
tok = AutoTokenizer.from_pretrained(MODEL)
tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(
    MODEL, torch_dtype=torch.bfloat16, device_map='auto',
    quantization_config=BitsAndBytesConfig(load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_quant_type='nf4'))
print(f'temel model: {sum(p.numel() for p in model.parameters())/1e9:.2f} MİLYAR parametre')

In [ ]:
# Eğitim örnekleri: sohbet şablonlu soru-cevap + derlemden kod blokları
from datasets import Dataset
def chatf(ins, out):
    return tok.apply_chat_template([
        {'role': 'system', 'content': 'Sen Vega, Türkçe konuşan kod ağırlıklı bir yapay zekâ asistanısın. Kısa, doğru ve teknik cevaplar verirsin.'},
        {'role': 'user', 'content': ins},
        {'role': 'assistant', 'content': out}], tokenize=False)
texts = [chatf(p['instruction'], p['output']) for p in pairs]
# derlemi 1500 karakterlik bloklara böl, örneklem al (dil modelleme sinyali)
blocks = [corpus[i:i+1500] for i in range(0, len(corpus), 1500)][::6][:1500]
texts += blocks
ds = Dataset.from_dict({'text': texts}).shuffle(seed=42)
def tokenize(b):
    t = tok(b['text'], truncation=True, max_length=512, padding='max_length')
    t['labels'] = t['input_ids'].copy()
    return t
ds = ds.map(tokenize, batched=True, remove_columns=['text'])
print(len(ds), 'eğitim örneği')

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, task_type='CAUSAL_LM',
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj']))
model.print_trainable_parameters()   # ~11M eğitilebilir / 1.24B toplam

In [ ]:
from transformers import TrainingArguments, Trainer
trainer = Trainer(model=model, train_dataset=ds, args=TrainingArguments(
    output_dir='vega-1b', per_device_train_batch_size=4,
    gradient_accumulation_steps=4, num_train_epochs=2,
    learning_rate=2e-4, lr_scheduler_type='cosine', warmup_ratio=0.03,
    bf16=True, logging_steps=20, save_strategy='epoch', report_to='none'))
trainer.train()   # T4: ~60-90 dk, A100: ~15-25 dk — kayıp eğrisi GERÇEK eğitimdir

In [ ]:
# Test: Vega-1B ile konuş
def sor(q, n=200):
    msgs = [{'role': 'system', 'content': 'Sen Vega, Türkçe konuşan kod ağırlıklı bir yapay zekâ asistanısın.'},
            {'role': 'user', 'content': q}]
    ids = tok.apply_chat_template(msgs, return_tensors='pt', add_generation_prompt=True).to(model.device)
    out = model.generate(ids, max_new_tokens=n, temperature=0.7, do_sample=True,
                         pad_token_id=tok.eos_token_id)
    print(tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True))
sor('Closure nedir? Kısaca açıkla.')
sor('Goroutine ile thread farkı nedir?')

In [ ]:
# Kaydet: LoRA adaptörü küçüktür (~45 MB) — Drive'a veya HF'e yükleyebilirsin
model.save_pretrained('vega-1b-lora')
tok.save_pretrained('vega-1b-lora')
!du -sh vega-1b-lora
# Yerelde çalıştırmak için: adaptörü birleştir + llama.cpp ile GGUF'a çevir →
# ollama/LM Studio'da 'vega-1b' olarak kullan. WebGPU (tarayıcı) için: MLC-LLM dönüşümü.